# Python Programming Test Solutions

**Test Date:** 18 February 2026  
**Source File:** `Python_Test1_18_Feb_2026.md`

This notebook contains clean solutions for all questions from Section A and Section B.

## Section A - Code Correction (40 Marks)

## A1. Stable Top-K Selection

Corrected function:
- does not mutate the input list
- preserves original order for ties

In [ ]:
def top_k(items, k):
    if k <= 0:
        return []

    indexed_items = list(enumerate(items))
    indexed_items.sort(key=lambda t: (-t[1][1], t[0]))
    return [item for _, item in indexed_items[:k]]


items = [("A", 90), ("B", 95), ("C", 95), ("D", 85)]
print(top_k(items, 2))

## A2. Category Word Counter

Bug fixed: `=+` replaced with proper increment logic.

In [ ]:
def count_by_category(pairs):
    result = {}
    for cat, word in pairs:
        if cat not in result:
            result[cat] = {}
        result[cat][word] = result[cat].get(word, 0) + 1
    return result


pairs = [("sports", "win"), ("sports", "win"), ("tech", "ai")]
print(count_by_category(pairs))

## A3. Missing Number (0 to n)

Bug fixed: integer arithmetic is used directly.

In [ ]:
def missing_number(nums):
    n = len(nums)
    total = n * (n + 1) // 2
    return total - sum(nums)


nums = [3, 0, 1]
print(missing_number(nums))

## A4. Default Argument Bug

Fix: use `None` as default and create a fresh list for each call.

In [ ]:
def add_unique(x, seen=None):
    if seen is None:
        seen = []
    if x not in seen:
        seen.append(x)
    return seen


print(add_unique(1))
print(add_unique(2))

## Section B - Code Completion (60 Marks)

## B5. Moving Average of Last w Elements

Time Complexity: O(n)

In [ ]:
from collections import deque


def moving_average(stream, w):
    if w <= 0:
        raise ValueError("Window size w must be positive")

    window = deque()
    running_sum = 0
    averages = []

    for value in stream:
        window.append(value)
        running_sum += value

        if len(window) > w:
            running_sum -= window.popleft()

        averages.append(running_sum / len(window))

    return averages


stream = [1, 2, 3, 4, 5]
print(moving_average(stream, 3))

## B6. Run-Length Decode with Escape Support

Rules handled:
- multi-digit counts
- escaped characters via backslash
- invalid format raises `ValueError`

In [ ]:
def rle_decode(s):
    i = 0
    out = []
    n = len(s)

    while i < n:
        if not s[i].isdigit():
            raise ValueError("Invalid format: count expected")

        count_start = i
        while i < n and s[i].isdigit():
            i += 1
        count = int(s[count_start:i])

        if i >= n:
            raise ValueError("Invalid format: missing symbol after count")

        if s[i] == chr(92):
            i += 1
            if i >= n:
                raise ValueError("Invalid format: dangling escape")
            ch = s[i]
            i += 1
        else:
            ch = s[i]
            i += 1

        out.append(ch * count)

    return "".join(out)


print(rle_decode("3a2b1c"))
print(rle_decode(r"2\3"))

## B7. Longest Unique Substring

Time Complexity: O(n)

In [ ]:
def longest_unique_substring(s):
    last_pos = {}
    left = 0
    best = 0

    for right, ch in enumerate(s):
        if ch in last_pos and last_pos[ch] >= left:
            left = last_pos[ch] + 1
        last_pos[ch] = right
        best = max(best, right - left + 1)

    return best


print(longest_unique_substring("abcabcbb"))

## B8. Balanced Brackets with Quotes

Rules handled:
- ignore brackets inside single or double quotes
- escaped quotes are supported
- validates (), {}, []

In [ ]:
def is_balanced_quoted(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    opening = set(pairs.values())
    stack = []

    in_quote = None
    escaped = False

    for ch in s:
        if escaped:
            escaped = False
            continue

        if ch == chr(92):
            escaped = True
            continue

        if in_quote is not None:
            if ch == in_quote:
                in_quote = None
            continue

        if ch in ("'", '"'):
            in_quote = ch
            continue

        if ch in opening:
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False

    if escaped:
        return False

    return in_quote is None and not stack


print(is_balanced_quoted('("(]")'))
print(is_balanced_quoted("([{}])"))
print(is_balanced_quoted("([)]"))

## B9. Merge Overlapping Intervals with IDs

Rules handled:
- touching intervals are merged
- IDs in each merged block are sorted

In [ ]:
def merge_intervals_with_ids(intervals):
    if not intervals:
        return []

    intervals_sorted = sorted(intervals, key=lambda t: (t[1], t[2], t[0]))

    merged = []
    cur_start = intervals_sorted[0][1]
    cur_end = intervals_sorted[0][2]
    cur_ids = [intervals_sorted[0][0]]

    for _id, start, end in intervals_sorted[1:]:
        if start <= cur_end:
            cur_end = max(cur_end, end)
            cur_ids.append(_id)
        else:
            merged.append((cur_start, cur_end, sorted(cur_ids)))
            cur_start, cur_end, cur_ids = start, end, [_id]

    merged.append((cur_start, cur_end, sorted(cur_ids)))
    return merged


intervals = [(1, 1, 3), (2, 2, 5), (3, 7, 8)]
print(merge_intervals_with_ids(intervals))

## B10. Robust File Data Parser

Efficient approach:
- single pass over data
- dictionary stores age, score sum, and score count per name
- inconsistent age entries for same name are ignored

In [ ]:
def parse(lines):
    records = {}

    for raw in lines:
        if not isinstance(raw, str):
            continue

        parts = [p.strip() for p in raw.split(",")]
        if len(parts) != 3:
            continue

        name, age_text, score_text = parts
        if not name:
            continue

        try:
            age = int(age_text)
            score = float(score_text)
        except ValueError:
            continue

        if not (1 <= age <= 120):
            continue
        if not (0 <= score <= 100):
            continue

        if name not in records:
            records[name] = [age, score, 1]
            continue

        saved_age, total, count = records[name]
        if age != saved_age:
            continue

        records[name] = [saved_age, total + score, count + 1]

    result = {}
    for name, (age, total, count) in records.items():
        result[name] = (age, total / count)

    return result


sample_lines = [
    "John,25,90",
    "John,25,80",
    "Anna,130,70",
    "Bob,20,105",
]

print(parse(sample_lines))

## Quick Validation

Basic checks against sample expectations.

In [ ]:
assert top_k([("A", 90), ("B", 95), ("C", 95), ("D", 85)], 2) == [("B", 95), ("C", 95)]
assert count_by_category([("sports", "win"), ("sports", "win"), ("tech", "ai")]) == {
    "sports": {"win": 2},
    "tech": {"ai": 1},
}
assert missing_number([3, 0, 1]) == 2
assert add_unique(1) == [1] and add_unique(2) == [2]
assert moving_average([1, 2, 3, 4, 5], 3) == [1.0, 1.5, 2.0, 3.0, 4.0]
assert rle_decode("3a2b1c") == "aaabbc"
assert rle_decode("2\\3") == "33"
assert longest_unique_substring("abcabcbb") == 3
assert is_balanced_quoted('("(]")') is True
assert merge_intervals_with_ids([(1, 1, 3), (2, 2, 5), (3, 7, 8)]) == [(1, 5, [1, 2]), (7, 8, [3])]
assert parse(["John,25,90", "John,25,80", "Anna,130,70", "Bob,20,105"]) == {"John": (25, 85.0)}

print("All sample checks passed.")